# 1 · Daten erkunden und bereinigen
**Päckli & Co.** · Data Science für Marketing · FHNW CAS Data Driven Marketing

Päckli & Co. ist ein Online-Shop für Geschenke und Wohnideen. Der Umsatz flacht ab, und viele früher aktive
Kundinnen und Kunden kaufen nicht mehr. Ihr seid das CRM-Team und habt eine Kampagne. Heute Morgen lernt ihr die
Daten kennen und macht sie sauber.

**Minimum:** eine saubere Bestelltabelle `orders_clean` und RFM-Werte pro Kunde (`rfm`), im Team-Ordner gespeichert.
**Erweiterung (optional):** Umsatz pro Land, Wiederkäufe, Kundensegmente.

## So funktioniert Colab
- Ein Notebook besteht aus **Zellen**. Textzellen erklären, Codezellen rechnen.
- **Ausführen:** auf ▶ links neben der Zelle klicken oder `Shift + Enter` drücken.
- **Gemini:** das Gemini-Symbol oben rechts öffnet den Chat. Gemini sieht euer Notebook und schreibt Code, den ihr
  mit einem Klick als neue Zelle einfügen könnt.
- **Fehlermeldung?** Gebt die ganze Meldung an Gemini: «Erkläre den Fehler und korrigiere den Code.»
- **Speichern:** zuerst *Datei > Kopie in Drive speichern*, sonst geht eure Arbeit verloren.

**Mit KI arbeiten: fünf Regeln**
1. Erst denken, dann prompten: Schritt für Schritt, eine Frage pro Prompt.
2. Beschreibt das Ziel in Business-Sprache, nicht in Code.
3. Lasst euch jede Abfrage in einem Satz erklären.
4. Prüft jede Zahl mit gesundem Menschenverstand.
5. Vertraut keinem Ergebnis, das ihr nicht erklären könnt.

**So arbeitet ihr in jedem Schritt**
1. Lest das **Ziel**.
2. Kopiert den **Prompt** ins Gemini-Fenster (Gemini-Symbol oben rechts) und lasst euch den Code erklären.
3. Fügt den Code in die leere Zelle ein und führt sie aus (▶ links oder `Shift + Enter`).
4. **Plausibel?** Prüft das Ergebnis mit gesundem Menschenverstand.
5. Steckt ihr fest, führt die **💡 Lösung** aus. Das ist keine Schande: Die Lösung liefert dieselben Tabellen wie euer Code.

In [ ]:
#@title ▶ Setup: tragt eure Teamnummer ein und führt diese Zelle zuerst aus
TEAM = 0  #@param {type:"integer"}
BASIS_URL = "https://raw.githubusercontent.com/badrustanicki/fhnw-ds-marketing-2026/main/"
import os, urllib.request
if not os.path.exists("kurs.py"):
    urllib.request.urlretrieve(BASIS_URL + "kurs.py", "kurs.py")
import duckdb, numpy as np, pandas as pd, matplotlib.pyplot as plt
from kurs import *
start(TEAM, BASIS_URL)

**Die Daten:** drei Tabellen in DuckDB, einer Datenbank direkt im Notebook. Abfragen schreibt ihr mit
`duckdb.sql("SELECT ...")`.

| Tabelle | Spalten |
|---|---|
| `orders` | `invoice` Rechnungsnummer, `stock_code` Produkt, `quantity` Menge, `invoice_date` Datum, `price` Stückpreis CHF, `customer_id` Kunde |
| `products` | `stock_code` Produkt, `description` Bezeichnung |
| `customers` | `customer_id` Kunde, `country` Land |

Wichtig: Eine Zeile in `orders` ist eine **Position auf einer Rechnung**, nicht eine Bestellung.

## Los geht's: die erste Frage an Gemini

### 0 Wie viele Bestellungen gibt es?

**Ziel:** Eine erste Frage an die Daten stellen und prüfen, ob die Antwort Sinn ergibt.

**Prompt für Gemini** (kopieren und im Gemini-Fenster einfügen):
```text
Wie viele Bestellungen gibt es? Die Daten liegen in DuckDB in der Tabelle orders. Schreibe Python-Code mit duckdb.sql(...) und erkläre die Abfrage in einem Satz.
```

In [ ]:
# Euer Code: fügt hier ein, was Gemini vorschlägt, und führt die Zelle aus (▶).

In [ ]:
#@title 💡 Lösung: nur wenn ihr nicht weiterkommt (Doppelklick zeigt den Code, ▶ führt ihn aus)
duckdb.sql("""
SELECT COUNT(*)                AS zeilen,
       COUNT(DISTINCT invoice) AS rechnungen
FROM orders
""")

**Plausibel?** Hat Gemini **Zeilen** oder **Rechnungen** gezählt? Eine Rechnung hat oft mehrere Zeilen. Welche Zahl beantwortet die Frage? Und zählen Stornos (Rechnungsnummer beginnt mit C) als Bestellungen?

*Funktioniert Gemini bei euch nicht?* Nehmt einen beliebigen Chatbot im Browser, gebt ihm die Tabellenbeschreibung oben mit und kopiert den Code hierher.

## Die Checkliste: Daten erkunden

### 1 Kontext und Datentypen

**Ziel:** Verstehen, was eine Zeile ist und welche Spalten welchen Typ haben.

**Prompt für Gemini** (kopieren und im Gemini-Fenster einfügen):
```text
Zeige mit duckdb.sql die ersten 10 Zeilen von orders und die Spalten mit Datentypen der Tabellen orders, products und customers. Was beschreibt eine Zeile in orders?
```

In [ ]:
# Euer Code: fügt hier ein, was Gemini vorschlägt, und führt die Zelle aus (▶).

In [ ]:
#@title 💡 Lösung: nur wenn ihr nicht weiterkommt (Doppelklick zeigt den Code, ▶ führt ihn aus)
display(duckdb.sql("DESCRIBE orders").df())
display(duckdb.sql("SELECT * FROM orders LIMIT 10").df())
display(duckdb.sql("SELECT * FROM products LIMIT 5").df())
display(duckdb.sql("SELECT * FROM customers LIMIT 5").df())

**Plausibel?** Eine Zeile ist eine Position: ein Produkt, eine Menge, ein Stückpreis auf einer Rechnung. Der Umsatz einer Zeile ist also `quantity * price`.

### 2 Fehlende Werte und Duplikate

**Ziel:** Herausfinden, wie viele Zeilen keinen Kunden haben und wie viele Zeilen doppelt vorkommen.

**Prompt für Gemini** (kopieren und im Gemini-Fenster einfügen):
```text
Wie viele Zeilen in orders haben keine customer_id? Wie viele Zeilen sind exakte Duplikate, also in allen Spalten gleich wie eine andere Zeile?
```

In [ ]:
# Euer Code: fügt hier ein, was Gemini vorschlägt, und führt die Zelle aus (▶).

In [ ]:
#@title 💡 Lösung: nur wenn ihr nicht weiterkommt (Doppelklick zeigt den Code, ▶ führt ihn aus)
duckdb.sql("""
SELECT COUNT(*)                                              AS zeilen,
       COUNT(*) - COUNT(customer_id)                         AS ohne_kunde,
       COUNT(*) - (SELECT COUNT(*) FROM (SELECT DISTINCT * FROM orders)) AS duplikate
FROM orders
""")

**Plausibel?** Zeilen ohne Kunde sind Gastbestellungen: Für RFM und Kampagnen sind sie nutzlos. Duplikate entstehen beim Laden von Daten und würden Umsatz doppelt zählen.

### 3 Stornos

**Ziel:** Stornierte Rechnungen erkennen.

**Prompt für Gemini** (kopieren und im Gemini-Fenster einfügen):
```text
Rechnungen, deren Nummer (invoice) mit C beginnt, sind Stornos. Wie viele Zeilen sind das? Wie viele Zeilen haben eine negative Menge (quantity)? Zeige ein paar Beispiele.
```

In [ ]:
# Euer Code: fügt hier ein, was Gemini vorschlägt, und führt die Zelle aus (▶).

In [ ]:
#@title 💡 Lösung: nur wenn ihr nicht weiterkommt (Doppelklick zeigt den Code, ▶ führt ihn aus)
display(duckdb.sql("""
SELECT COUNT(*) FILTER (WHERE invoice LIKE 'C%') AS storno_zeilen,
       COUNT(*) FILTER (WHERE quantity < 0)      AS negative_mengen
FROM orders
""").df())
duckdb.sql("SELECT * FROM orders WHERE invoice LIKE 'C%' LIMIT 5")

**Plausibel?** Stornos haben negative Mengen. Zählt man sie mit, sinkt der Umsatz; zählt man sie als Bestellungen, steigt die Zahl der Bestellungen. Beides ist falsch.

### 4 Top-Produkte

**Ziel:** Die meistverkauften Produkte finden. Frage der Kursleitung um 11 Uhr: Was ist euer Top-Produkt, und ist es ein Produkt?

**Prompt für Gemini** (kopieren und im Gemini-Fenster einfügen):
```text
Welche 10 Produkte kommen in orders auf den meisten verschiedenen Rechnungen vor? Zeige stock_code, description aus products und die Anzahl Rechnungen.
```

In [ ]:
# Euer Code: fügt hier ein, was Gemini vorschlägt, und führt die Zelle aus (▶).

In [ ]:
#@title 💡 Lösung: nur wenn ihr nicht weiterkommt (Doppelklick zeigt den Code, ▶ führt ihn aus)
duckdb.sql("""
SELECT o.stock_code, p.description, COUNT(DISTINCT o.invoice) AS rechnungen
FROM orders o JOIN products p ON p.stock_code = o.stock_code
GROUP BY o.stock_code, p.description
ORDER BY rechnungen DESC
LIMIT 10
""")

**Plausibel?** Steht oben «Porto» oder «Versandkosten»? Das sind keine Produkte. Sucht alle solchen Codes: `duckdb.sql("SELECT * FROM products WHERE stock_code NOT LIKE 'P1%'")`. Achtung: `NOT LIKE 'P%'` würde «POST» übersehen.

In [ ]:
duckdb.sql("SELECT * FROM products WHERE stock_code NOT LIKE 'P1%'")

### 5 Auffällige Kunden und Mengen

**Ziel:** Kunden und Bestellungen finden, die nicht ins Bild passen.

**Prompt für Gemini** (kopieren und im Gemini-Fenster einfügen):
```text
Welche 10 Kunden haben in orders die meisten Rechnungen? Zeige pro Kunde die Anzahl Rechnungen, die Anzahl Zeilen, die durchschnittliche Menge und den durchschnittlichen Preis. Zeige ausserdem die 5 Zeilen mit der grössten Menge (quantity).
```

In [ ]:
# Euer Code: fügt hier ein, was Gemini vorschlägt, und führt die Zelle aus (▶).

In [ ]:
#@title 💡 Lösung: nur wenn ihr nicht weiterkommt (Doppelklick zeigt den Code, ▶ führt ihn aus)
display(duckdb.sql("""
SELECT customer_id, COUNT(DISTINCT invoice) AS rechnungen, COUNT(*) AS zeilen,
       ROUND(AVG(quantity), 1) AS menge_avg, ROUND(AVG(price), 2) AS preis_avg
FROM orders WHERE customer_id IS NOT NULL
GROUP BY customer_id ORDER BY rechnungen DESC LIMIT 10
""").df())
duckdb.sql("SELECT * FROM orders ORDER BY quantity DESC LIMIT 5")

**Plausibel?** Ein Kunde mit Hunderten Rechnungen zu je einem Stück für ein paar Rappen? Das ist ein internes Testkonto. Eine Bestellung über 80'000 Stück, die kurz danach storniert wurde? Ein Tippfehler.

### 6 Preise und Daten über die Zeit

**Ziel:** Prüfen, ob Preise und Datumsangaben über die ganze Zeit plausibel sind.

**Prompt für Gemini** (kopieren und im Gemini-Fenster einfügen):
```text
Zeige den Median von price pro Monat (aus invoice_date) als Tabelle und als Liniendiagramm. Welches ist das früheste und das späteste Datum in orders?
```

In [ ]:
# Euer Code: fügt hier ein, was Gemini vorschlägt, und führt die Zelle aus (▶).

In [ ]:
#@title 💡 Lösung: nur wenn ihr nicht weiterkommt (Doppelklick zeigt den Code, ▶ führt ihn aus)
preise = duckdb.sql("""
SELECT strftime(invoice_date, '%Y-%m') AS monat, MEDIAN(price) AS preis_median, COUNT(*) AS zeilen
FROM orders GROUP BY monat ORDER BY monat
""").df()
display(preise)
preise.plot(x="monat", y="preis_median", marker="o", figsize=(10, 3), legend=False, title="Median-Preis pro Monat (CHF)")
plt.show()
duckdb.sql("SELECT MIN(invoice_date) AS erstes, MAX(invoice_date) AS letztes FROM orders")

**Plausibel?** Ein Monat mit 100-mal höheren Preisen? Dort wurden die Preise in Rappen statt in Franken geladen. Und ein Datum im Jahr 2052? Die Daten reichen nur bis zum 31. August 2025: ein Tippfehler.

## Die saubere Tabelle

### 7 orders_clean erstellen

**Ziel:** Eine Tabelle ohne die Probleme aus den Schritten 2 bis 6. Mit ihr arbeitet ihr den ganzen Tag.

**Prompt für Gemini** (kopieren und im Gemini-Fenster einfügen):
```text
Erstelle in DuckDB eine Tabelle orders_clean aus orders: ohne exakte Duplikate, ohne Stornos (invoice beginnt mit C) und ohne negative Mengen, ohne Zeilen ohne customer_id, ohne das Testkonto K00001, ohne die Nicht-Produkte POST, DOT, BANK, M und S, ohne Daten nach dem 31. August 2025 und ohne Mengen ab 10000. Im Juni 2024 sind die Preise in Rappen: dort price durch 100 teilen. Vergleiche danach den Umsatz (quantity * price) von orders und orders_clean.
```

In [ ]:
# Euer Code: fügt hier ein, was Gemini vorschlägt, und führt die Zelle aus (▶).

In [ ]:
#@title 💡 Lösung: nur wenn ihr nicht weiterkommt (Doppelklick zeigt den Code, ▶ führt ihn aus)
duckdb.sql("""
CREATE OR REPLACE TABLE orders_clean AS
SELECT DISTINCT
       invoice, stock_code, quantity, invoice_date,
       CASE WHEN strftime(invoice_date, '%Y-%m') = '2024-06' THEN price / 100 ELSE price END AS price,
       customer_id
FROM orders
WHERE invoice NOT LIKE 'C%' AND quantity > 0 AND quantity < 10000   -- Stornos, Riesenbestellung
  AND customer_id IS NOT NULL AND customer_id <> 'K00001'           -- Gäste, Testkonto
  AND stock_code NOT IN ('POST', 'DOT', 'BANK', 'M', 'S')           -- keine Produkte
  AND invoice_date < DATE '2025-09-01'                              -- Tippfehler 2052
""")
duckdb.sql("""
SELECT (SELECT ROUND(SUM(quantity * price)) FROM orders)       AS umsatz_roh,
       (SELECT ROUND(SUM(quantity * price)) FROM orders_clean) AS umsatz_sauber
""")

**Plausibel?** Wie viel Umsatz bleibt übrig? Erinnert euch an die Folie «Wenn die KI sich irrt»: Welche Zahl hätte die Geschäftsleitung bekommen? Die Prüfung unten sucht die häufigsten Probleme.

In [ ]:
plausibel("orders_clean")

### 8 Umsatz pro Monat

**Ziel:** Saisonalität sehen: Wann verkauft Päckli & Co. am meisten?

**Prompt für Gemini** (kopieren und im Gemini-Fenster einfügen):
```text
Zeichne den Umsatz (quantity * price) pro Monat aus orders_clean als Balkendiagramm.
```

In [ ]:
# Euer Code: fügt hier ein, was Gemini vorschlägt, und führt die Zelle aus (▶).

In [ ]:
#@title 💡 Lösung: nur wenn ihr nicht weiterkommt (Doppelklick zeigt den Code, ▶ führt ihn aus)
umsatz_monat = duckdb.sql("""
SELECT strftime(invoice_date, '%Y-%m') AS monat, SUM(quantity * price) AS umsatz
FROM orders_clean GROUP BY monat ORDER BY monat
""").df()
umsatz_monat.plot.bar(x="monat", y="umsatz", figsize=(10, 3), legend=False, title="Umsatz pro Monat (CHF)")
plt.show()

**Plausibel?** Gibt es einen Höhepunkt vor Weihnachten? Sieht der Juni 2024 jetzt normal aus? Der letzte Monat ist August 2025: Danach ist die Zukunft, die ihr nicht seht.

## Das Ergebnis: RFM pro Kunde

### 9 RFM-Werte

**Ziel:** Für jeden Kunden drei Zahlen: Wie lange ist der letzte Kauf her (Recency), wie oft hat er bestellt (Frequency), wie viel Umsatz (Monetary)? Stichtag ist der 31. August 2025.

**Prompt für Gemini** (kopieren und im Gemini-Fenster einfügen):
```text
Berechne aus orders_clean pro Kunde: tage_seit_kauf (Tage vom letzten Kauf bis zum 31. August 2025), frequenz (Anzahl verschiedene Rechnungen) und umsatz (Summe quantity * price). Speichere das Ergebnis als DataFrame rfm mit den Spalten customer_id, tage_seit_kauf, frequenz, umsatz.
```

In [ ]:
# Euer Code: fügt hier ein, was Gemini vorschlägt, und führt die Zelle aus (▶).

In [ ]:
#@title 💡 Lösung: nur wenn ihr nicht weiterkommt (Doppelklick zeigt den Code, ▶ führt ihn aus)
rfm = duckdb.sql("""
SELECT customer_id,
       date_diff('day', MAX(invoice_date)::DATE, DATE '2025-08-31') AS tage_seit_kauf,
       COUNT(DISTINCT invoice)                                      AS frequenz,
       ROUND(SUM(quantity * price), 2)                              AS umsatz
FROM orders_clean
GROUP BY customer_id
ORDER BY umsatz DESC
""").df()
rfm.head(10)

**Plausibel?** Gute Kunden mit vielen Tagen seit dem letzten Kauf sind die Zielgruppe des Tages. Gibt es negative Tage (Datum nach dem Stichtag)? Absurde Umsätze (Rappen-Monat)?

In [ ]:
plausibel("rfm", rfm)

### 10 Speichern
Speichert beide Ergebnisse in eurem Team-Ordner auf Google Drive, damit die nächsten Notebooks sie laden können.
Beim ersten Mal fragt Colab, ob es auf Google Drive zugreifen darf: euer Konto wählen und zulassen.
Euer Team-Ordner muss als Verknüpfung in «Meine Ablage» liegen (Anleitung von der Kursleitung).

In [ ]:
speichern("orders_clean", "orders_clean")
speichern(rfm, "rfm")

## Erweiterung (optional)

### E1 Umsatz pro Land

**Ziel:** Woher kommen die Kunden, woher der Umsatz?

**Prompt für Gemini** (kopieren und im Gemini-Fenster einfügen):
```text
Zeige aus orders_clean und customers den Umsatz und die Anzahl Kunden pro Land, absteigend nach Umsatz.
```

In [ ]:
# Euer Code: fügt hier ein, was Gemini vorschlägt, und führt die Zelle aus (▶).

In [ ]:
#@title 💡 Lösung: nur wenn ihr nicht weiterkommt (Doppelklick zeigt den Code, ▶ führt ihn aus)
duckdb.sql("""
SELECT c.country AS land, COUNT(DISTINCT o.customer_id) AS kunden, ROUND(SUM(o.quantity * o.price)) AS umsatz
FROM orders_clean o JOIN customers c ON c.customer_id = o.customer_id
GROUP BY land ORDER BY umsatz DESC
""")

**Plausibel?** Ist Päckli & Co. ein Schweizer Shop? Lohnen sich Kampagnen im Ausland?

### E2 Wiederkäufe

**Ziel:** Wie viele Kunden kommen wieder, und wie lange dauert es bis zur nächsten Bestellung?

**Prompt für Gemini** (kopieren und im Gemini-Fenster einfügen):
```text
Wie viele Kunden in orders_clean haben genau einmal, zwei- bis viermal und öfter bestellt? Wie viele Tage liegen im Median zwischen zwei Bestellungen desselben Kunden?
```

In [ ]:
# Euer Code: fügt hier ein, was Gemini vorschlägt, und führt die Zelle aus (▶).

In [ ]:
#@title 💡 Lösung: nur wenn ihr nicht weiterkommt (Doppelklick zeigt den Code, ▶ führt ihn aus)
display(duckdb.sql("""
SELECT CASE WHEN frequenz = 1 THEN '1' WHEN frequenz <= 4 THEN '2 bis 4' ELSE '5 und mehr' END AS bestellungen,
       COUNT(*) AS kunden
FROM rfm GROUP BY bestellungen ORDER BY bestellungen
""").df())
duckdb.sql("""
WITH r AS (SELECT DISTINCT customer_id, invoice, MIN(invoice_date) AS datum FROM orders_clean GROUP BY customer_id, invoice),
     z AS (SELECT date_diff('day', LAG(datum) OVER (PARTITION BY customer_id ORDER BY datum), datum) AS tage FROM r)
SELECT MEDIAN(tage) AS median_tage_zwischen_bestellungen FROM z WHERE tage IS NOT NULL
""")

**Plausibel?** Wenn der Median-Abstand mehrere Wochen ist: Ab wann ist ein Kunde «still»?

### E3 Kundensegmente

**Ziel:** Kunden mit k-Means in vier Gruppen teilen (unüberwachtes Lernen).

**Prompt für Gemini** (kopieren und im Gemini-Fenster einfügen):
```text
Teile die Kunden in rfm mit k-Means in 4 Segmente. Nimm den Logarithmus (np.log1p) von tage_seit_kauf, frequenz und umsatz und standardisiere sie. Zeige pro Segment die Anzahl Kunden und die Mittelwerte der drei Spalten.
```

In [ ]:
# Euer Code: fügt hier ein, was Gemini vorschlägt, und führt die Zelle aus (▶).

In [ ]:
#@title 💡 Lösung: nur wenn ihr nicht weiterkommt (Doppelklick zeigt den Code, ▶ führt ihn aus)
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
X = StandardScaler().fit_transform(np.log1p(rfm[["tage_seit_kauf", "frequenz", "umsatz"]]))
rfm_seg = rfm.assign(segment=KMeans(n_clusters=4, n_init=10, random_state=0).fit_predict(X))
rfm_seg.groupby("segment").agg(kunden=("customer_id", "size"), tage_seit_kauf=("tage_seit_kauf", "mean"),
                               frequenz=("frequenz", "mean"), umsatz=("umsatz", "mean")).round(0)

**Plausibel?** Könnt ihr jedem Segment einen Namen geben (z. B. «treue Grosskunden», «eingeschlafen»)? Wenn nicht, sind die Segmente vielleicht nicht nützlich.